# Compare ScintPi Products with Commerical ISMR Data

This tutorial runs ScintPi raw files through `scintkit`, then compares the derived `sigma_phi_1` product with `ref_ismr`, a commercial sensor (Septentrio PolaRx5S) reference dataset.

More specifically this example uses data recording during a rare mid-latitude scintillation event that was captured in parallel by the commerical sensor and a low-cost ScintPi 3.0 sensor at the same time. 

Start from a repository checkout where you have followed the README installation step, for example `python -m pip install -e .` from the repository root.

In [ ]:

import sys

from pathlib import Path

import numpy as np
import pandas as pd

import scintkit


Use paths relative to the repository root. Start Jupyter from the repo root, the same place you run the README install command.

In [ ]:
data_dir = Path("data")
raw_files = sorted(data_dir.glob("scintpi3_*lvl0.pq"))
#These can also be .bin or .bin.zip files

#lets only choose one file for testing, but you can try all others too.
raw_files = raw_files

if not raw_files:
    raise FileNotFoundError(
        f"No ScintPi files found in {data_dir.resolve()}. "
        "Start Jupyter from the repository root, or update data_dir."
    )

raw_files

scintkit.pipelines.auto.process contains a wrapper that combines several steps of the process to produce lvl3 1-minute rate scintillation index files. It can accept a list of files including '.bin.zip', '.bin' or 'lvl0.pq' files.

In [ ]:
lvl3_files = scintkit.pipelines.auto.process(raw_files, verbose=True)
lvl3_files

Load and combine the ScintPi level-3 products.

In [ ]:
scintpi_df = pd.concat(
    (pd.read_parquet(path) for path in lvl3_files),
    ignore_index=True,
)

scintpi_df.head()

Load the reference ISMR dataset from the commercial sensor.

In [ ]:
ismr_path = data_dir / "ref_ismr_oct11_0to18.pq"
ismr_df = pd.read_parquet(ismr_path)

ismr_df.head()

Merge both datasets by one-minute time bin and satellite PRN.

In [ ]:
merged_df = pd.merge(scintpi_df, ismr_df, on=["minbin", "prn"], how="inner")

merged_df[["minbin", "prn", "sigma_phi_1", "sigma_phi_1_ismr"]].he

Apply a simple quality mask before comparing scintillation products.

In [ ]:
mask = (
    (merged_df["elev"] > 30)
    & (merged_df["n_1"] > 590)
    & (merged_df["quality_1"] == 0)
    & (merged_df["n_cycleslip_1"] < 10)
)

comparison_df = merged_df.loc[mask].copy()
comparison_df.shape

Plot ScintPi `sigma_phi_1` against the reference ISMR value. To quantify the fit, we use $R^2$ statistics.

In [ ]:
import matplotlib.pyplot as plt

x = comparison_df["sigma_phi_1"].to_numpy()
y = comparison_df["sigma_phi_1_ismr"].to_numpy()

ss_res = np.sum((y - x) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - (ss_res / ss_tot)
n = len(comparison_df)
max_val = max(np.nanmax(x), np.nanmax(y))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x, y, alpha=0.5, s=20, c="tab:blue", label="Data points")
ax.plot([0, max_val], [0, max_val], "r--", alpha=1, label="1:1 line")
ax.text(
    0.05,
    0.95,
    f"N={n}\n$R^2$={r2:.2f}",
    transform=ax.transAxes,
    va="top",
    fontsize=12,
    bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.5},
)
ax.set(
    xlabel="sigma_phi_1 (ScintPi)",
    ylabel="sigma_phi_1 (reference ISMR)",
    xlim=(0, 1),
    ylim=(0, 1),
    title="sigma_phi_1 comparison, October 11, 2024",
)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()